In [ ]:
# %pip install pandas pyarrow FlagEmbedding qdrant_client

In [ ]:
#Note: Used google colab's A100 GPU runtime for generating embeddings
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import pandas as pd
from FlagEmbedding import BGEM3FlagModel

In [ ]:
import torch
print(torch.__version__)
print(torch.version.hip)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
description_parquet_in = "/content/drive/MyDrive/470data/descriptions.parquet"
review_parquet_in = "/content/drive/MyDrive/470data/reviews.parquet"
description_embedding_out = "/content/drive/MyDrive/470data/descriptions.parquet"
review_embedding_out = "/content/drive/MyDrive/470data/descriptions.parquet"

description_embedding_col = ["appid", "merged_description"]
review_embedding_col = ["appid"] + [f"review_{i}" for i in range(1,11)]

In [ ]:
df_desciption = pd.read_parquet(description_parquet_in)
df_review = pd.read_parquet(review_parquet_in)

description_data = df_desciption[description_embedding_col]
review_data = df_review[review_embedding_col]

In [ ]:
model = BGEM3FlagModel(
    'BAAI/bge-m3',
    devices='cuda:0',
    use_fp16=True
)
print(f"Model device: {model.target_devices}")

In [ ]:
def embedDescription():
    output = model.encode(
        description_data["merged_description"].to_list(),
        batch_size=64,
        max_length=8192,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )
    description_embeddings = output["dense_vecs"]
    appids = description_data["appid"].to_list()
    assert len(appids) == description_embeddings.shape[0]

    df_out = pd.DataFrame({
        "appid": appids,
        "embedding": list(description_embeddings)  # convert 2D array -> list of 1D arrays
    })
    df_out.to_parquet(description_embedding_out, index=False)

    print(df_out.shape)
    df_out.head()


In [ ]:
def embedReview():
    review_cols = [f"review_{i}" for i in range(1, 11)]
    df_out = pd.DataFrame({"appid": review_data["appid"].to_list()})

    for col in review_cols:
        series = review_data[col]

        mask = series.notna() & (series.astype(str).str.strip() != "")

        texts = series[mask].to_list()

        if len(texts) == 0:
            df_out[f"{col}_embedding"] = [None] * len(df_out)
            continue

        output = model.encode(
            texts,
            batch_size=64,
            max_length=8192,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )
        embeddings = output["dense_vecs"] 

        col_embeddings = [None] * len(series)
        emb_iter = iter(embeddings.tolist())
        for idx, is_valid in enumerate(mask):
            if is_valid:
                col_embeddings[idx] = next(emb_iter)

        df_out[f"{col}_embedding"] = col_embeddings

    df_out.to_parquet(review_embedding_out, index=False)
    print(df_out.shape)
    df_out.head()

In [ ]:
embedDescription()

In [ ]:
embedReview()